# 4D-OBC height-threshold sensitivity analysis

This notebook evaluates how the `RegionGrowingAlgorithm.height_threshold` affects 4D Objects-by-Change extracted from the mesh-derived reference time series.

For every configured height threshold it:

1. uses the same mesh epochs, reference epoch, corepoints and signed Z-displacement matrix;
2. runs the complete 4D-OBC extraction in an independent archive;
3. summarizes object count, seed count, spatial coverage, duration and spatiotemporal support;
4. compares the union of all detected object supports with a configurable baseline threshold using global TP/FP/FN, precision, recall, F1 and IoU;
5. performs Hungarian one-to-one object matching to quantify object-level stability;
6. plots sensitivity curves, object-property distributions, temporal activity, spatial support and support-difference maps.

An object has fixed spatial support over its lifetime. Its spatiotemporal support is therefore the Cartesian product of its corepoint set and continuous timestamp interval. Support units are **corepoint-days**.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

import py4dgeo
from py4dgeo.data_loader import read_obj_and_assign_timestamps
from py4dgeo.segmentation import RegionGrowingAlgorithm

pd.set_option("display.max_columns", None)

## 1. Configuration

In [ ]:
# Input mesh series
data_path = Path(r'C:\rsa\research_proj\blender_project\simulation_test2\data\sceneparts\simulation_test2')
output_dir = Path.cwd() / "objects_parameter_sensitivity_results"
output_dir.mkdir(parents=True, exist_ok=True)

# Mesh timestamps and common reference/corepoints
reference_timestamp = datetime(2020, 1, 7, 0, 0, 0)
start_timestamp = datetime(2020, 1, 1, 0, 0, 0)
time_increment = timedelta(days=1)
vertex_voxel_size = 1.0  # None = uses all reference vertices

# Fixed 4D-OBC parameters
obc_neighborhood_radius = 1.0
obc_min_segments = 10
obc_minperiod = 3
obc_thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
obc_seed_subsampling = 1

# Parameter under investigation [m]
height_thresholds = [0.02, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20]
baseline_height_threshold = 0.05

# Object matching across threshold runs. This removes zero/near-zero Hungarian pairs.
object_match_iou_threshold = 0.01

# Plot/export controls
map_point_size = 4
map_columns = 3
save_figures = True
figure_dpi = 300

height_thresholds = sorted({float(value) for value in height_thresholds})
if any(value <= 0 for value in height_thresholds):
    raise ValueError("All height_thresholds must be positive.")
if not any(np.isclose(baseline_height_threshold, value) for value in height_thresholds):
    raise ValueError("baseline_height_threshold must be included in height_thresholds.")
baseline_height_threshold = next(value for value in height_thresholds if np.isclose(value, baseline_height_threshold))

print("Height thresholds [m]:", height_thresholds)
print("Baseline threshold [m]:", baseline_height_threshold)
print("Output directory:", output_dir)

## 2. Load the reference mesh series and define common corepoints

In [ ]:
epochs = read_obj_and_assign_timestamps(str(data_path), start_timestamp, time_increment)
if not epochs:
    raise RuntimeError(f"No epochs loaded. Check data_path: {data_path}")

reference_epoch, other_epochs = py4dgeo.extract_reference_and_others(epochs, reference_timestamp)
all_ref_vertices = np.asarray(reference_epoch.cloud)

if vertex_voxel_size is None:
    vertex_indices = np.arange(len(all_ref_vertices), dtype=int)
else:
    voxel_keys = np.floor(all_ref_vertices / float(vertex_voxel_size)).astype(np.int64)
    _, first_indices = np.unique(voxel_keys, axis=0, return_index=True)
    vertex_indices = np.sort(first_indices)

corepoints = np.ascontiguousarray(all_ref_vertices[vertex_indices], dtype=np.float64)
target_timestamps = [epoch.timestamp for epoch in other_epochs]
timedeltas = [timestamp - reference_epoch.timestamp for timestamp in target_timestamps]
time_days = np.array([(timestamp - target_timestamps[0]).total_seconds() / 86400.0 for timestamp in target_timestamps])

print(f"Loaded {len(epochs)} epochs: {epochs[0].timestamp} -> {epochs[-1].timestamp}")
print(f"Reference epoch: {reference_epoch.timestamp}")
print(f"Corepoints: {len(all_ref_vertices):,} -> {len(corepoints):,}")
print(f"Analyzed target epochs: {len(other_epochs)}")

In [ ]:
print(f"Reference epoch: {reference_epoch.timestamp}")
print(
    f"Corepoints: {len(all_ref_vertices):,} "
    f"-> {len(corepoints):,}"
)
print(f"Analyzed target epochs: {len(other_epochs)}")
print(
    f"Target timestamp range: "
    f"{target_timestamps[0]} -> {target_timestamps[-1]}"
)

## 3. Calculate signed Z-displacements once

In [ ]:
ref_z = all_ref_vertices[vertex_indices, 2]
distances = np.full((len(corepoints), len(other_epochs)), np.nan, dtype=np.float64)

for column, target_epoch in enumerate(tqdm(other_epochs, desc="Signed Z displacement")):
    target_vertices = np.asarray(target_epoch.cloud)
    if len(target_vertices) != len(all_ref_vertices):
        print(f"Warning: {target_epoch.timestamp} has {len(target_vertices):,} vertices; expected {len(all_ref_vertices):,}. Skipped.")
        continue
    distances[:, column] = target_vertices[vertex_indices, 2] - ref_z

if not np.isfinite(distances).any():
    raise RuntimeError("The displacement matrix contains no finite values.")

print("Distance matrix:", distances.shape)
print(f"Range: {np.nanmin(distances):.4f} to {np.nanmax(distances):.4f} m")

## 4. Run an independent 4D-OBC extraction for every height threshold

In [ ]:
uncertainty_dtype = [
    ("lodetection", "<f8"), ("spread1", "<f8"), ("num_samples1", "<i8"),
    ("spread2", "<f8"), ("num_samples2", "<i8"),
]

def _threshold_tag(value):
    return f"{float(value):.4f}".rstrip("0").rstrip(".").replace(".", "p")

def _new_reference_analysis(height_threshold):
    archive_path = output_dir / f"reference_height_threshold_{_threshold_tag(height_threshold)}m.zip"
    analysis = py4dgeo.SpatiotemporalAnalysis(str(archive_path), force=True)
    analysis.reference_epoch = reference_epoch
    analysis.corepoints = corepoints
    analysis.timedeltas = timedeltas
    analysis.distances = distances
    analysis.m3c2 = None
    analysis.uncertainties = np.full(distances.shape, np.nan, dtype=uncertainty_dtype)
    algorithm = RegionGrowingAlgorithm(
        neighborhood_radius=obc_neighborhood_radius,
        min_segments=obc_min_segments,
        minperiod=obc_minperiod,
        height_threshold=float(height_threshold),
        thresholds=obc_thresholds,
        seed_subsampling=obc_seed_subsampling,
    )
    analysis.invalidate_results(seeds=True, objects=True, smoothed_distances=False)
    objects = list(algorithm.run(analysis))
    return analysis, objects, archive_path

analyses_by_threshold = {}
objects_by_threshold = {}
archive_paths = {}

for height_threshold in height_thresholds:
    print()
    print(f"Running height_threshold={height_threshold:g} m")
    analysis, objects, archive_path = _new_reference_analysis(height_threshold)
    analyses_by_threshold[height_threshold] = analysis
    objects_by_threshold[height_threshold] = objects
    archive_paths[height_threshold] = archive_path
    print(f"Seeds={len(analysis.seeds):,}; objects={len(objects):,}; saved={archive_path}")

## 5. Object and union-support metrics

For a corepoint (c), overlapping object time intervals are merged before global support is counted. Thus, overlapping objects do not double-count the same corepoint-time support.

For comparison with the baseline threshold:

\[
\mathrm{IoU}=\frac{TP}{TP+FP+FN},\quad
\mathrm{precision}=\frac{TP}{TP+FP},\quad
\mathrm{recall}=\frac{TP}{TP+FN}.
\]

In [ ]:
IOU_EPS = 1e-12

def _object_properties(obj):
    indices = np.unique(np.asarray(obj.indices, dtype=int))
    start_epoch, end_epoch = int(obj.start_epoch), int(obj.end_epoch)
    if start_epoch < 0 or end_epoch >= len(time_days) or start_epoch > end_epoch:
        raise ValueError(f"Invalid object interval [{start_epoch}, {end_epoch}]")
    start_time, end_time = float(time_days[start_epoch]), float(time_days[end_epoch])
    duration = end_time - start_time
    return {
        "indices": indices, "index_set": set(map(int, indices)),
        "start_epoch": start_epoch, "end_epoch": end_epoch,
        "start_time": start_time, "end_time": end_time,
        "duration_days": duration, "n_corepoints": len(indices),
        "support_corepoint_days": len(indices) * duration,
    }

def _merge_intervals(intervals):
    merged = []
    for start, end in sorted(intervals):
        if not merged or start > merged[-1][1] + IOU_EPS:
            merged.append([float(start), float(end)])
        else:
            merged[-1][1] = max(merged[-1][1], float(end))
    return [(start, end) for start, end in merged]

def _support_by_corepoint(objects):
    support = {}
    for obj in objects:
        props = _object_properties(obj)
        if props["duration_days"] <= 0:
            continue
        interval = (props["start_time"], props["end_time"])
        for index in props["indices"]:
            support.setdefault(int(index), []).append(interval)
    return {index: _merge_intervals(intervals) for index, intervals in support.items()}

def _interval_duration(intervals):
    return float(sum(end - start for start, end in intervals))

def _shared_duration(left, right):
    total = 0.0
    i = j = 0
    while i < len(left) and j < len(right):
        total += max(0.0, min(left[i][1], right[j][1]) - max(left[i][0], right[j][0]))
        if left[i][1] <= right[j][1]: i += 1
        else: j += 1
    return total

def _global_support_metrics(candidate_objects, baseline_objects):
    candidate = _support_by_corepoint(candidate_objects)
    baseline = _support_by_corepoint(baseline_objects)
    indices = set(candidate) | set(baseline)
    candidate_total = sum(_interval_duration(candidate.get(i, [])) for i in indices)
    baseline_total = sum(_interval_duration(baseline.get(i, [])) for i in indices)
    tp = sum(_shared_duration(candidate.get(i, []), baseline.get(i, [])) for i in indices)
    fp, fn = max(0.0, candidate_total - tp), max(0.0, baseline_total - tp)
    precision = tp / (tp + fp) if tp + fp > 0 else np.nan
    recall = tp / (tp + fn) if tp + fn > 0 else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else np.nan
    iou = tp / (tp + fp + fn) if tp + fp + fn > 0 else np.nan
    return {"global_tp": tp, "global_fp": fp, "global_fn": fn,
            "global_precision": precision, "global_recall": recall,
            "global_f1": f1, "global_iou": iou}

def _pairwise_st_iou(left_obj, right_obj):
    left, right = _object_properties(left_obj), _object_properties(right_obj)
    shared_corepoints = len(left["index_set"] & right["index_set"])
    overlap_duration = max(0.0, min(left["end_time"], right["end_time"]) - max(left["start_time"], right["start_time"]))
    intersection = shared_corepoints * overlap_duration
    union = left["support_corepoint_days"] + right["support_corepoint_days"] - intersection
    return intersection / union if union > 0 else 0.0

def _hungarian_object_metrics(candidate_objects, baseline_objects):
    matrix = np.zeros((len(candidate_objects), len(baseline_objects)), dtype=float)
    for i, candidate in enumerate(candidate_objects):
        for j, baseline in enumerate(baseline_objects):
            matrix[i, j] = _pairwise_st_iou(candidate, baseline)
    if matrix.size:
        rows, columns = linear_sum_assignment(1.0 - matrix)
        matched_ious = matrix[rows, columns]
        keep = matched_ious >= object_match_iou_threshold
        rows, columns, matched_ious = rows[keep], columns[keep], matched_ious[keep]
    else:
        rows = columns = np.array([], dtype=int)
        matched_ious = np.array([], dtype=float)
    n_matches = len(matched_ious)
    precision = n_matches / len(candidate_objects) if candidate_objects else np.nan
    recall = n_matches / len(baseline_objects) if baseline_objects else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else np.nan
    return {
        "n_object_matches": n_matches,
        "object_precision": precision, "object_recall": recall, "object_f1": f1,
        "mean_matched_st_iou": np.mean(matched_ious) if n_matches else np.nan,
        "median_matched_st_iou": np.median(matched_ious) if n_matches else np.nan,
    }, pd.DataFrame({"candidate_object_index": rows, "baseline_object_index": columns, "st_iou": matched_ious})

def _support_duration_vector(objects):
    values = np.zeros(len(corepoints), dtype=float)
    for index, intervals in _support_by_corepoint(objects).items():
        values[index] = _interval_duration(intervals)
    return values

## 6. Build summary and comparison tables

In [ ]:
baseline_objects = objects_by_threshold[baseline_height_threshold]
summary_rows, object_rows, match_tables = [], [], {}

for height_threshold in height_thresholds:
    objects = objects_by_threshold[height_threshold]
    properties = [_object_properties(obj) for obj in objects]
    support_vector = _support_duration_vector(objects)
    global_metrics = _global_support_metrics(objects, baseline_objects)
    object_metrics, match_table = _hungarian_object_metrics(objects, baseline_objects)
    match_table.insert(0, "height_threshold_m", height_threshold)
    match_tables[height_threshold] = match_table

    for object_index, props in enumerate(properties):
        object_rows.append({
            "height_threshold_m": height_threshold, "object_index": object_index,
            "n_corepoints": props["n_corepoints"], "start_epoch": props["start_epoch"],
            "end_epoch": props["end_epoch"], "duration_days": props["duration_days"],
            "support_corepoint_days": props["support_corepoint_days"],
        })

    summary_rows.append({
        "height_threshold_m": height_threshold,
        "is_baseline": np.isclose(height_threshold, baseline_height_threshold),
        "n_seeds": len(analyses_by_threshold[height_threshold].seeds),
        "n_objects": len(objects),
        "unique_object_corepoints": int(np.count_nonzero(support_vector > 0)),
        "corepoint_coverage_pct": 100 * np.mean(support_vector > 0),
        "union_support_corepoint_days": np.sum(support_vector),
        "summed_object_support_corepoint_days": np.sum([p["support_corepoint_days"] for p in properties]),
        "median_object_corepoints": np.median([p["n_corepoints"] for p in properties]) if properties else np.nan,
        "median_object_duration_days": np.median([p["duration_days"] for p in properties]) if properties else np.nan,
        **global_metrics, **object_metrics,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("height_threshold_m").reset_index(drop=True)
object_details_df = pd.DataFrame(object_rows)
object_matches_df = pd.concat(match_tables.values(), ignore_index=True) if match_tables else pd.DataFrame()

summary_df["object_count_change_vs_baseline"] = summary_df["n_objects"] - len(baseline_objects)
summary_df["object_count_change_vs_baseline_pct"] = 100 * summary_df["object_count_change_vs_baseline"] / len(baseline_objects) if baseline_objects else np.nan
baseline_union_support = float(summary_df.loc[summary_df["is_baseline"], "union_support_corepoint_days"].iloc[0])
summary_df["union_support_change_vs_baseline_pct"] = 100 * (summary_df["union_support_corepoint_days"] - baseline_union_support) / baseline_union_support if baseline_union_support else np.nan

display(summary_df.round(3))
display(object_details_df.groupby("height_threshold_m")[["n_corepoints", "duration_days", "support_corepoint_days"]].describe().round(2))

summary_df.to_csv(output_dir / "height_threshold_summary.csv", index=False)
object_details_df.to_csv(output_dir / "object_details.csv", index=False)
object_matches_df.to_csv(output_dir / "object_matches_to_baseline.csv", index=False)
print("Saved CSV tables to", output_dir)

## 7. Sensitivity curves

In [ ]:
x = summary_df["height_threshold_m"].to_numpy()
fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)

axes[0, 0].plot(x, summary_df["n_objects"], "o-", label="Objects")
axes[0, 0].plot(x, summary_df["n_seeds"], "s--", label="Seeds")
axes[0, 0].set_ylabel("Count"); axes[0, 0].legend()

axes[0, 1].plot(x, summary_df["corepoint_coverage_pct"], "o-", color="tab:green", label="Corepoint coverage")
axes[0, 1].set_ylabel("Covered corepoints [%]")
ax_support = axes[0, 1].twinx()
ax_support.plot(x, summary_df["union_support_corepoint_days"], "s--", color="tab:purple", label="Union support")
ax_support.set_ylabel("Union support [corepoint-days]")
lines = axes[0, 1].lines + ax_support.lines
axes[0, 1].legend(lines, [line.get_label() for line in lines], loc="best")

axes[1, 0].plot(x, summary_df["global_iou"], "o-", label="Global IoU")
axes[1, 0].plot(x, summary_df["global_precision"], "s--", label="Global precision")
axes[1, 0].plot(x, summary_df["global_recall"], "^--", label="Global recall")
axes[1, 0].plot(x, summary_df["global_f1"], "d-.", label="Global F1")
axes[1, 0].set_ylabel("Union-support agreement"); axes[1, 0].set_ylim(0, 1.03); axes[1, 0].legend()

axes[1, 1].plot(x, summary_df["mean_matched_st_iou"], "o-", label="Mean matched ST-IoU")
axes[1, 1].plot(x, summary_df["object_f1"], "s--", label="Object F1")
axes[1, 1].set_ylabel("Object-level agreement"); axes[1, 1].set_ylim(0, 1.03); axes[1, 1].legend()

for ax in axes.flat:
    ax.axvline(baseline_height_threshold, color="black", linestyle=":", linewidth=1, label="Baseline")
    ax.set_xlabel("Height threshold [m]"); ax.grid(alpha=0.25)
fig.suptitle("4D-OBC height-threshold sensitivity", fontweight="bold")
if save_figures: fig.savefig(output_dir / "sensitivity_curves.png", dpi=figure_dpi, bbox_inches="tight")
plt.show()

## 8. Object-property distributions

In [ ]:
if object_details_df.empty:
    print("No objects were extracted.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    fields = [("n_corepoints", "Object size [corepoints]"), ("duration_days", "Duration [days]"), ("support_corepoint_days", "ST support [corepoint-days]")]
    labels = [f"{value:g}" for value in height_thresholds]
    colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(height_thresholds)))
    for ax, (field, ylabel) in zip(axes, fields):
        data = [object_details_df.loc[object_details_df["height_threshold_m"] == value, field].dropna().to_numpy() for value in height_thresholds]
        box = ax.boxplot(data, labels=labels, widths=0.5, patch_artist=True, showfliers=False, showmeans=True,
                         meanprops={"marker": "^", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 5})
        for patch, color in zip(box["boxes"], colors): patch.set_facecolor(color); patch.set_alpha(0.8)
        ax.set_xlabel("Height threshold [m]"); ax.set_ylabel(ylabel); ax.grid(axis="y", alpha=0.25)
    fig.suptitle("Object-property distributions", fontweight="bold")
    if save_figures: fig.savefig(output_dir / "object_property_distributions.png", dpi=figure_dpi, bbox_inches="tight")
    plt.show()

## 9. Temporal distribution of objects

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
for height_threshold in height_thresholds:
    active = np.zeros(len(time_days), dtype=int)
    for obj in objects_by_threshold[height_threshold]:
        props = _object_properties(obj)
        active[props["start_epoch"]:props["end_epoch"] + 1] += 1
    ax.plot(target_timestamps, active, marker="o", markersize=3, linewidth=1.3, label=f"{height_threshold:g} m")
ax.set_xlabel("Time"); ax.set_ylabel("Number of active objects")
ax.set_title("Temporal distribution of active 4D-OBCs", fontweight="bold")
ax.grid(alpha=0.25); ax.legend(title="Height threshold")
fig.autofmt_xdate()
if save_figures: fig.savefig(output_dir / "temporal_object_distribution.png", dpi=figure_dpi, bbox_inches="tight")
plt.show()

## 10. Spatial distribution of object support

In [ ]:
n_panels = len(height_thresholds)
n_columns = min(map_columns, n_panels)
n_rows = int(np.ceil(n_panels / n_columns))
support_vectors = {value: _support_duration_vector(objects_by_threshold[value]) for value in height_thresholds}
vmax = max((np.nanmax(values) for values in support_vectors.values()), default=1.0)
vmax = vmax if np.isfinite(vmax) and vmax > 0 else 1.0

fig, axes = plt.subplots(n_rows, n_columns, figsize=(4.6 * n_columns, 4.1 * n_rows), squeeze=False, constrained_layout=True)
scatter = None
for ax, height_threshold in zip(axes.flat, height_thresholds):
    values = support_vectors[height_threshold]
    scatter = ax.scatter(corepoints[:, 0], corepoints[:, 1], c=values, s=map_point_size, cmap="viridis", vmin=0, vmax=vmax, linewidths=0)
    ax.set_title(f"height threshold = {height_threshold:g} m" + chr(10) + f"objects = {len(objects_by_threshold[height_threshold])}")
    ax.set_xlabel("X [m]"); ax.set_ylabel("Y [m]"); ax.set_aspect("equal"); ax.grid(alpha=0.15)
for ax in axes.flat[n_panels:]: ax.set_visible(False)
if scatter is not None: fig.colorbar(scatter, ax=list(axes.flat[:n_panels]), shrink=0.85, label="Union object support duration [days]")
fig.suptitle("Spatial distribution of 4D-OBC support", fontweight="bold")
if save_figures: fig.savefig(output_dir / "spatial_object_support.png", dpi=figure_dpi, bbox_inches="tight")
plt.show()

## 11. Spatial support differences from the baseline threshold

Positive values mean that the tested threshold classifies a corepoint as object support for more days than the baseline. Negative values mean that support is lost relative to the baseline.

In [ ]:
difference_thresholds = [value for value in height_thresholds if not np.isclose(value, baseline_height_threshold)]
if not difference_thresholds:
    print("No non-baseline thresholds to compare.")
else:
    baseline_support = support_vectors[baseline_height_threshold]
    differences = {value: support_vectors[value] - baseline_support for value in difference_thresholds}
    max_abs_difference = max((np.nanmax(np.abs(values)) for values in differences.values()), default=1.0)
    max_abs_difference = max_abs_difference if np.isfinite(max_abs_difference) and max_abs_difference > 0 else 1.0
    n_columns = min(map_columns, len(difference_thresholds)); n_rows = int(np.ceil(len(difference_thresholds) / n_columns))
    fig, axes = plt.subplots(n_rows, n_columns, figsize=(4.6 * n_columns, 4.1 * n_rows), squeeze=False, constrained_layout=True)
    scatter = None
    for ax, height_threshold in zip(axes.flat, difference_thresholds):
        values = differences[height_threshold]
        scatter = ax.scatter(corepoints[:, 0], corepoints[:, 1], c=values, s=map_point_size, cmap="coolwarm",
                             norm=mcolors.TwoSlopeNorm(vmin=-max_abs_difference, vcenter=0, vmax=max_abs_difference), linewidths=0)
        changed_pct = 100 * np.mean(np.abs(values) > IOU_EPS)
        ax.set_title(f"{height_threshold:g} m minus {baseline_height_threshold:g} m" + chr(10) + f"changed corepoints = {changed_pct:.1f}%")
        ax.set_xlabel("X [m]"); ax.set_ylabel("Y [m]"); ax.set_aspect("equal"); ax.grid(alpha=0.15)
    for ax in axes.flat[len(difference_thresholds):]: ax.set_visible(False)
    if scatter is not None: fig.colorbar(scatter, ax=list(axes.flat[:len(difference_thresholds)]), shrink=0.85, label="Object-support duration difference [days]")
    fig.suptitle("Difference from baseline object support", fontweight="bold")
    if save_figures: fig.savefig(output_dir / "spatial_support_difference_from_baseline.png", dpi=figure_dpi, bbox_inches="tight")
    plt.show()

## 12. Object start-time and duration distribution

In [ ]:
if not object_details_df.empty:
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    colors = dict(zip(height_thresholds, plt.cm.viridis(np.linspace(0.15, 0.85, len(height_thresholds)))))
    for height_threshold in height_thresholds:
        subset = object_details_df[object_details_df["height_threshold_m"] == height_threshold]
        jitter = np.linspace(-0.12, 0.12, len(subset)) if len(subset) > 1 else np.zeros(len(subset))
        ax.scatter(subset["start_epoch"] + jitter, subset["duration_days"], s=22, alpha=0.65,
                   color=colors[height_threshold], label=f"{height_threshold:g} m")
    ax.set_xlabel("Object start epoch"); ax.set_ylabel("Object duration [days]")
    ax.set_title("Object temporal-position distribution", fontweight="bold")
    ax.grid(alpha=0.25); ax.legend(title="Height threshold", ncol=min(5, len(height_thresholds)))
    if save_figures: fig.savefig(output_dir / "object_start_duration_distribution.png", dpi=figure_dpi, bbox_inches="tight")
    plt.show()

## Interpretation checklist

- A stable parameter range should show a plateau in object count, union support and global IoU/F1.
- Large object-count changes with high global IoU usually indicate mostly split/merge changes rather than a different overall change support.
- A fall in global precision at a lower height threshold indicates newly included support outside the baseline; a fall in recall at a higher threshold indicates lost baseline support.
- Global support agreement can remain high while mean matched object IoU falls, revealing changed object identities or boundaries.
- Inspect the spatial difference maps to determine whether sensitivity is concentrated near object boundaries or changes entire regions.

The baseline is a comparison anchor, not ground truth. Repeat the analysis with another baseline if the conclusion depends strongly on that choice.